In [1]:
import os
import pickle
import faiss
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer
import requests
import re
from dotenv import load_dotenv


/home/ianli/anaconda3/envs/ecommerce_llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Paths
INDEX_PATH = "../index/product.index"
MAPPING_PATH = "../index/id_to_filename.pkl"
DOCUMENTS_DIR = "../outputs"
TOP_K = 5

In [3]:

# Load FAISS index and mapping
index = faiss.read_index(INDEX_PATH)
with open(MAPPING_PATH, "rb") as f:
    id_to_filename = pickle.load(f)

# Validate mapping size matches index
assert len(id_to_filename) == index.ntotal, "Mismatch FAISS vectors and ID map length"


# Build chunk size and offsets
chunk_sizes = {}
file_offsets = {}
sorted_chunk_files = sorted(set(id_to_filename.values()))
offset = 0
for file in sorted_chunk_files:
    df = pd.read_parquet(os.path.join(DOCUMENTS_DIR, file), columns=["text"])
    size = len(df)
    chunk_sizes[file] = size
    file_offsets[file] = offset
    offset += size

assert sum(chunk_sizes.values()) == index.ntotal, "Mismatch FAISS vectors and parquet rows"

# Load embedding model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
model.eval()

/home/ianli/anaconda3/envs/ecommerce_llm/lib/python3.11/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/ianli/anaconda3/envs/ecommerce_llm/lib/python3.11/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False})
  (2): Normalize()
)

In [5]:

def find_file_and_local_idx(global_idx):
    for file in sorted_chunk_files:
        start = file_offsets[file]
        end = start + chunk_sizes[file]
        if start <= global_idx < end:
            return file, global_idx - start
    raise IndexError(f"Global index {global_idx} not found in any file.")

def extract_summary(text):
    match = re.search(r"Summary:\s*(.*)", text)
    return match.group(1).strip() if match else ""

def remove_embedded_summary(text):
    return re.sub(r"Summary:\s*.*", "", text).strip()

def retrieve_top_k_reviews(query, top_k=TOP_K):
    query_emb = model.encode([query], convert_to_tensor=False).astype("float32")
    D, I = index.search(query_emb, top_k)
    results = []

    for global_idx, distance in zip(I[0], D[0]):
        file, local_idx = find_file_and_local_idx(global_idx)
        df = pd.read_parquet(os.path.join(DOCUMENTS_DIR, file))

        if local_idx >= len(df):
            continue  # Skip corrupted/missing index

        row = df.iloc[local_idx]
        metadata = row.get('metadata', {}) if 'metadata' in row else {}

        raw_text = row.get('text', '')
        summary = extract_summary(raw_text)
        text = remove_embedded_summary(raw_text)
        if not text:
            continue

        results.append({
            "asin": metadata.get('asin', 'N/A'),
            "rating": metadata.get('overall', 'N/A'),
            "votes": metadata.get('votes', 'N/A'),
            "vote_bin": metadata.get('vote_bin', 'N/A'),
            "verified": metadata.get('verified', 'N/A'),
            "reviewer": metadata.get('reviewerName', 'N/A'),
            "date": metadata.get('reviewTime', 'N/A'),
            "summary": summary,
            "text": text,
            "distance": float(distance),
            "similarity_score": float(distance)
        })
    return results

def display_retrieval_results(query, retrieved_reviews):
    print(f"\n=== FAISS Retrieval Results for Query: '{query}' ===")
    print(f"Found {len(retrieved_reviews)} results:\n")
    for i, r in enumerate(retrieved_reviews):
        print(f"Result {i+1}:")
        print(f"  Similarity Score: {r['similarity_score']:.4f}")
        print(f"  ASIN: {r.get('asin', 'N/A')}")
        print(f"  Rating: {r.get('rating', 'N/A')} stars")
        print(f"  Verified: {r.get('verified', 'N/A')}")
        if r.get('summary'):
            print(f"  Summary: {r['summary']}")
        print(f"  Text Preview: {r.get('text', '')[:100]}...")
        print("-" * 50)

In [6]:

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

def call_groq_llm(prompt, model="llama3-70b-8192", temperature=0.7, max_tokens=512):
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {os.getenv('GROQ_API_KEY')}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model,
        "messages": [
            {"role": "system", "content": "You are a helpful e-commerce assistant."},
            {"role": "user", "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }
    response = requests.post(url, headers=headers, json=data)
    if response.status_code == 200:
        return response.json()['choices'][0]['message']['content']
    else:
        raise Exception(f"Groq API error {response.status_code}: {response.text}")

In [12]:

def build_rag_prompt(query, retrieved_reviews):
    context = ""
    for i, r in enumerate(retrieved_reviews):
        context += f"Review {i+1} (Similarity: {r['similarity_score']:.4f}):\n"
        if r.get('summary'):
            context += f"Summary: {r['summary']}\n"
        if r.get('rating') != 'N/A' or r.get('verified') != 'N/A':
            context += f"Rating: {r.get('rating', 'N/A')} stars | Verified: {r.get('verified', 'N/A')}\n"
        if r.get('votes') != 'N/A':
            context += f"Helpful Votes: {r['votes']}\n"
        if r.get('date') != 'N/A' or r.get('reviewer') != 'N/A':
            context += f"Date: {r.get('date', 'N/A')} | Reviewer: {r.get('reviewer', 'N/A')}\n"
        context += f"Text: {r.get('text', '')}\n\n"

    return f"""
You are a helpful e-commerce assistant. Use the following product reviews to answer the user's question.
The reviews are ranked by similarity to your query (higher similarity = more relevant).

--- START OF REVIEWS ---
{context.strip()}
--- END OF REVIEWS ---

User question:
{query}

Answer concisely, referencing the relevant reviews if needed. If the answer isn't clear, say so honestly.
""".strip()

# Example usage
query = "Does this phone have good battery life and camera quality?"
reviews = retrieve_top_k_reviews(query)
rag_prompt = build_rag_prompt(query, reviews)
print("RAG Prompt:\n", rag_prompt)
answer = call_groq_llm(rag_prompt)
print("\nGenerated Answer:\n", answer)

def print_similarity_summary(retrieved_reviews):
    print("\n=== Similarity Score Summary ===")
    df_summary = pd.DataFrame([
        {
            'Rank': i+1,
            'Similarity': r['similarity_score'],
            'ASIN': r.get('asin', 'N/A'),
            'Rating': r.get('rating', 'N/A'),
            'Summary': r.get('summary', '')[:50] + '...' if r.get('summary') else 'N/A'
        }
        for i, r in enumerate(retrieved_reviews)
    ])
    print(df_summary.to_string(index=False))

print_similarity_summary(reviews)

RAG Prompt:
 You are a helpful e-commerce assistant. Use the following product reviews to answer the user's question.
The reviews are ranked by similarity to your query (higher similarity = more relevant).

--- START OF REVIEWS ---
Review 1 (Similarity: 0.7270):
Summary: Five Stars
Rating: 5.0 stars | Verified: True
Helpful Votes: None
Date: 11 21, 2014 | Reviewer: John Bailey
Text: The battery life on this camera is amazing and the picture quality is great

Review 2 (Similarity: 0.7200):
Summary: Great phone needs better battery
Rating: 4.0 stars | Verified: True
Helpful Votes: 6.0
Date: 08 4, 2006 | Reviewer: CHRISTOPHERLEWIS
Text: I love this phone nice screen quality comfortable feel inyour hands for a 1.3 meg camera picture quality is great. only drawback is the battery life nokia needs to release anew and improve battery i am enjoying my phone it appeals to my friends

Review 3 (Similarity: 0.7104):
Summary: Three Stars
Rating: 3.0 stars | Verified: True
Helpful Votes: None
Date: